In [1]:
from langchain import hub
from langchain_ollama import ChatOllama
from langchain_core.tools import tool
from langchain.agents import create_react_agent, AgentExecutor
from langchain_community.tools import DuckDuckGoSearchRun
import requests

/Users/triptibhardwaj/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:34: NotOpenSSLWarning: urllib3 v2.0 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/triptibhardwaj/Library/Python/3.9/lib/python/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
llm = ChatOllama(
    model="qwen2.5:7b",
    temperature=0
)

In [3]:
search_tool = DuckDuckGoSearchRun()

In [4]:
@tool
def get_weather_data(city: str) -> str:
  """
  This function fetches the current weather data for a given city
  """
  url = f'https://api.weatherstack.com/current?access_key=51c8009f200d846f5e990a9d25318527&query={city}'
  response = requests.get(url)
  return response.json()

In [5]:
prompt = hub.pull("hwchase17/react")

/var/folders/n5/sj9n_6w138g50dzz75czm6hh0000gn/T/ipykernel_56180/4130516561.py:1: LangChainPendingDeprecationWarning: langchain.hub.pull is deprecated. Use the LangSmith SDK instead.
  prompt = hub.pull("hwchase17/react")
/Users/triptibhardwaj/Library/Python/3.9/lib/python/site-packages/langsmith/client.py:7618: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit list of allowed classes (or 'messages' for untrusted input that contains only chat messages) to suppress this warning.
  prompt = loads(json.dumps(prompt_object.manifest))


In [6]:
agent = create_react_agent(
    llm=llm,
    tools=[search_tool, get_weather_data],
    prompt=prompt
)

In [7]:
agent_executor = AgentExecutor(
    agent=agent,
    tools=[search_tool, get_weather_data],
    verbose=True,
    handle_parsing_errors=True
)

In [8]:
response = agent_executor.invoke({"input": "Find the capital of Madhya Pradesh, then find it's current weather condition"})



> Entering new AgentExecutor chain...
I need to first find the capital of Madhya Pradesh.
Action: duckduckgo_search
Action Input: capital of Madhya PradeshThe area covered by the present-day Madhya Pradesh includes the area of the ancient Avanti Mahajanapada , whose capital Ujjain (also known as ... Less-fertile red-to-yellow soils are spread over much of eastern Madhya Pradesh . ... Madhya Pradesh has a number of national parks and many wildlife ... Madhya Pradesh was created in 1950 from the former British Central Provinces and Berar and the princely states of Makrai and Chhattisgarh, with Nagpur ... Madhya Pradesh state is made up of 48 districts, which are grouped into nine divisions: Bhopal , Chambal, Gwalior , Hoshangabad, Indore , Jabalpur ... Madhya Pradesh state is established on 1-Nov-1956 and the capital of Madhya Pradesh is Bhopal. ... Total number of households in Madhya Pradesh State ...I now know that the capital of Madhya Pradesh is Bhopal.
Action: get_weather_data
Ac

In [9]:
response['output']

"The capital of Madhya Pradesh is Bhopal. Currently, the temperature in Bhopal is 27°C with partly cloudy skies. The wind speed is 15 km/h from the southwest (SW), and the humidity is at 73%. There is no precipitation reported. The air quality index is good with a US-EPA index of 2 and GB-Defra index of 2, indicating minimal pollution. The visibility is 10 kilometers, and it's currently nighttime."